# Implementasi Pertemuan 2
## Bridging Global Infrastructure to Address Local Challenges in High Performance Computing (HPC)

**Narasumber:** Dr. Warawan Dias Carbalo — Thammasat University, Thailand  
**Platform:** Python (Google Colab / Jupyter Notebook)

### Tujuan pembelajaran
Notebook ini menerapkan gagasan utama webinar: memanfaatkan komputasi berkinerja tinggi (*High Performance Computing/HPC*) untuk membantu menjawab masalah lokal. Contoh implementasi menggunakan **simulasi risiko genangan/banjir sederhana** pada wilayah hipotetis.

Setelah menyelesaikan notebook, mahasiswa diharapkan mampu:
1. Mengubah masalah lokal menjadi masalah komputasi yang terukur.
2. Membuat dan memvisualisasikan data spasial sintetis.
3. Menjalankan simulasi beberapa skenario curah hujan.
4. Mengukur waktu eksekusi dan memahami manfaat komputasi vektor.
5. Memahami konsep pekerjaan komputasi pada klaster HPC dan contoh skrip SLURM.
6. Menafsirkan hasil simulasi dengan memperhatikan batasan model.

> **Penting:** Ini adalah model pembelajaran yang sangat disederhanakan, bukan model hidrologi ilmiah. Data, elevasi, curah hujan, dan hasilnya sintetis. Hasil tidak boleh digunakan untuk peringatan banjir atau keputusan keselamatan nyata.

## 1. Memuat pustaka

Kita menggunakan NumPy untuk perhitungan numerik, Pandas untuk menyimpan hasil, Matplotlib untuk visualisasi, dan `time` untuk pengukuran waktu eksekusi.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("Pustaka berhasil dimuat.")

## 2. Merumuskan masalah lokal menjadi masalah komputasi

Webinar menekankan bahwa HPC akan bermanfaat ketika persoalan nyata diterjemahkan menjadi kebutuhan komputasi yang jelas. Contoh banjir dapat dirumuskan sebagai pertanyaan:

**Bagaimana perubahan curah hujan memengaruhi potensi genangan pada wilayah hipotetis?**

Untuk latihan ini, wilayah direpresentasikan sebagai kisi (*grid*) 2D. Setiap sel memiliki nilai elevasi. Hujan menambah volume air, sedangkan sel yang lebih rendah dianggap lebih rentan mengalami genangan. Kita membandingkan beberapa skenario curah hujan dan menghitung proporsi sel yang melewati ambang genangan sederhana.

Dalam riset sebenarnya, diperlukan data elevasi terverifikasi, infiltrasi tanah, drainase, aliran antar-sel, kondisi sungai, serta kalibrasi dan validasi dengan kejadian nyata.

## 3. Membuat data wilayah sintetis

Kita membangun grid elevasi hipotetis berukuran 100 × 100. Nilai elevasi dibuat dengan kombinasi kemiringan dan variasi acak agar menyerupai permukaan yang tidak rata.

In [ ]:
grid_size = 100
x = np.linspace(0, 1, grid_size)
y = np.linspace(0, 1, grid_size)
X, Y = np.meshgrid(x, y)

# Elevasi sintetis: kemiringan umum ditambah variasi permukaan.
elevation = (
    30
    + 12 * X
    + 8 * Y
    + 2.0 * np.sin(3 * np.pi * X) * np.cos(2 * np.pi * Y)
    + rng.normal(0, 0.7, size=(grid_size, grid_size))
)

# Wilayah dengan elevasi relatif rendah dianggap lebih rentan dalam model sederhana.
low_elevation_threshold = np.percentile(elevation, 25)
low_area_mask = elevation <= low_elevation_threshold

print("Ukuran grid:", elevation.shape)
print(f"Elevasi minimum: {elevation.min():.2f}")
print(f"Elevasi maksimum: {elevation.max():.2f}")
print(f"Ambang 25 persentil: {low_elevation_threshold:.2f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(elevation, origin="lower", cmap="terrain")
plt.colorbar(label="Elevasi sintetis (satuan relatif)")
plt.title("Peta Elevasi Hipotetis")
plt.xlabel("Kolom grid")
plt.ylabel("Baris grid")
plt.show()

plt.figure(figsize=(8, 6))
plt.imshow(low_area_mask, origin="lower", cmap="Blues")
plt.title("Area Elevasi Rendah (25% Terendah)")
plt.xlabel("Kolom grid")
plt.ylabel("Baris grid")
plt.show()

## 4. Simulasi satu skenario curah hujan

Untuk demonstrasi, kita menggunakan pendekatan sederhana:

- `rainfall_mm`: curah hujan pada suatu skenario.
- `runoff_factor`: bagian hujan yang diasumsikan menjadi limpasan.
- `relative_water_level`: indikator air relatif yang dihitung dari limpasan dan elevasi.
- `flood_threshold`: ambang indikator untuk menandai potensi genangan.

Model ini **tidak** menghitung aliran air secara fisik dari satu sel ke sel lain. Karena itu, hasilnya hanya untuk belajar tentang simulasi dan analisis skenario.

In [ ]:
runoff_factor = 0.65
flood_threshold = 18.0

def simulate_flood_indicator(elevation_grid, rainfall_mm,
                             runoff_factor=0.65, threshold=18.0):
    """Menghasilkan indikator genangan sintetis, bukan prediksi banjir nyata."""
    runoff = rainfall_mm * runoff_factor
    # Elevasi yang lebih rendah memperoleh indikator air relatif lebih besar.
    relative_water_level = runoff - (elevation_grid - elevation_grid.min()) * 0.35
    flood_mask = relative_water_level >= threshold
    return relative_water_level, flood_mask

rainfall_example = 60
water_level, flood_mask = simulate_flood_indicator(
    elevation, rainfall_example, runoff_factor, flood_threshold
)

flooded_cells = int(flood_mask.sum())
flooded_pct = flooded_cells / flood_mask.size * 100
print(f"Skenario hujan: {rainfall_example} mm")
print(f"Sel melewati ambang indikator: {flooded_cells:,} dari {flood_mask.size:,}")
print(f"Proporsi area pada grid: {flooded_pct:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(water_level, origin="lower", cmap="viridis")
axes[0].set_title("Indikator Ketinggian Air Relatif")
axes[0].set_xlabel("Kolom grid")
axes[0].set_ylabel("Baris grid")
plt.colorbar(im0, ax=axes[0], label="Nilai relatif")

axes[1].imshow(flood_mask, origin="lower", cmap="Blues")
axes[1].set_title("Sel yang Melewati Ambang")
axes[1].set_xlabel("Kolom grid")
axes[1].set_ylabel("Baris grid")

plt.tight_layout()
plt.show()

## 5. Membandingkan beberapa skenario hujan

Analisis skenario membantu menjawab pertanyaan “apa yang terjadi jika curah hujan meningkat?”. Kita mencoba beberapa tingkat curah hujan dan membandingkan proporsi sel yang melewati ambang indikator.

In [ ]:
rainfall_scenarios = [20, 40, 60, 80, 100, 120]
scenario_rows = []

for rain in rainfall_scenarios:
    level, mask = simulate_flood_indicator(
        elevation, rain, runoff_factor=runoff_factor, threshold=flood_threshold
    )
    scenario_rows.append({
        "curah_hujan_mm": rain,
        "jumlah_sel_terindikasi": int(mask.sum()),
        "proporsi_area_pct": mask.mean() * 100,
        "rata_rata_indikator": float(level.mean())
    })

scenario_results = pd.DataFrame(scenario_rows)
display(scenario_results.round(2))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    scenario_results["curah_hujan_mm"],
    scenario_results["proporsi_area_pct"],
    marker="o"
)
plt.title("Hubungan Curah Hujan dan Proporsi Area pada Ambang Indikator")
plt.xlabel("Curah hujan (mm)")
plt.ylabel("Proporsi area (%)")
plt.grid(True, alpha=0.3)
plt.show()

### Interpretasi
Jika proporsi area yang melewati ambang meningkat seiring bertambahnya hujan, model menunjukkan hubungan yang diharapkan dari asumsi yang dibuat. Namun, hasil ini tidak membuktikan bahwa suatu wilayah nyata akan mengalami banjir pada curah hujan tertentu. Ambang dan persamaan sengaja disederhanakan agar alur komputasi mudah dipelajari.

## 6. Mengapa HPC dapat dibutuhkan?

HPC relevan ketika simulasi perlu dilakukan pada grid yang jauh lebih besar, menggunakan banyak skenario, resolusi waktu tinggi, atau model fisik yang kompleks. Sebagai contoh:
- Grid 100 × 100 memiliki 10.000 sel.
- Grid 2.000 × 2.000 memiliki 4.000.000 sel.
- Ribuan skenario curah hujan dan parameter dapat memperbesar kebutuhan komputasi.

Kita memperlihatkan operasi array menggunakan NumPy. Operasi vektor biasanya lebih efisien daripada melakukan perhitungan satu per satu menggunakan loop Python. Ini bukan benchmark HPC yang sesungguhnya, tetapi memperkenalkan prinsip mengurangi beban komputasi yang tidak perlu.

In [ ]:
# Bandingkan perhitungan sederhana dengan loop dan operasi vektor.
benchmark_size = 500_000
values = rng.random(benchmark_size)

start = time.perf_counter()
loop_result = np.empty(benchmark_size)
for i in range(benchmark_size):
    loop_result[i] = values[i] * 0.65 - 3.0
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
vector_result = values * 0.65 - 3.0
vector_seconds = time.perf_counter() - start

print(f"Waktu loop Python : {loop_seconds:.6f} detik")
print(f"Waktu NumPy vektor: {vector_seconds:.6f} detik")
if vector_seconds > 0:
    print(f"Rasio waktu loop / vektor: {loop_seconds / vector_seconds:.2f}x")
print("Hasil sama:", np.allclose(loop_result, vector_result))

### Catatan benchmark
Waktu dapat berbeda menurut komputer, beban sistem, dan versi pustaka. Jangan menyimpulkan bahwa semua pekerjaan otomatis lebih cepat dengan cara yang sama. Pada HPC, peningkatan performa juga dipengaruhi komunikasi antarproses, akses penyimpanan, ukuran masalah, dan cara pembagian pekerjaan.

## 7. Simulasi Monte Carlo sederhana

Salah satu alasan menggunakan infrastruktur komputasi besar adalah menjalankan banyak simulasi dengan variasi parameter. Pada contoh ini, kita mengacak curah hujan untuk melihat sebaran hasil dari banyak skenario. Implementasi menggunakan operasi array NumPy agar efisien pada komputer biasa; pada skala jauh lebih besar, skenario independen dapat dibagi ke beberapa proses atau node HPC.

In [ ]:
n_simulations = 2000
rainfall_samples = rng.uniform(20, 120, n_simulations)

# Banyak simulasi dijalankan sekaligus; hasil tetap memakai model indikator sederhana.
simulated_area_pct = np.empty(n_simulations)
for i, rain in enumerate(rainfall_samples):
    _, mask = simulate_flood_indicator(
        elevation, rain, runoff_factor=runoff_factor, threshold=flood_threshold
    )
    simulated_area_pct[i] = mask.mean() * 100

mc_summary = pd.Series(simulated_area_pct).describe(
    percentiles=[0.05, 0.50, 0.95]
)
display(mc_summary.to_frame("persentase_area"))

plt.figure(figsize=(8, 5))
plt.hist(simulated_area_pct, bins=30)
plt.title("Distribusi Hasil Simulasi Monte Carlo")
plt.xlabel("Proporsi area pada ambang indikator (%)")
plt.ylabel("Frekuensi")
plt.show()

## 8. Contoh alur kerja pada klaster HPC

Webinar menjelaskan bahwa pemanfaatan HPC bukan hanya menulis kode. Pengguna juga perlu memahami akun, lingkungan Linux, terminal/SSH, penyimpanan, batas sumber daya, dan sistem antrean pekerjaan.

Alur umum:
1. Menyiapkan kode dan data.
2. Masuk ke sistem melalui SSH atau portal pengguna yang disediakan fasilitas.
3. Memuat lingkungan/perangkat lunak sesuai dokumentasi fasilitas.
4. Menulis skrip pekerjaan dan meminta sumber daya yang diperlukan.
5. Mengirim pekerjaan ke sistem antrean.
6. Memantau status, membaca log, lalu memeriksa hasil.
7. Mendokumentasikan konfigurasi agar eksperimen dapat diulang.

Perintah dan konfigurasi sebenarnya berbeda pada setiap fasilitas HPC. Ikuti dokumentasi dan kebijakan fasilitas yang digunakan.

### Contoh perintah terminal (ilustrasi)

```bash
# Hubungkan ke akun pada fasilitas yang telah diberikan aksesnya
ssh username@alamat-klaster

# Periksa direktori dan berkas
pwd
ls -lh

# Kirim pekerjaan SLURM
sbatch flood_job.slurm

# Periksa antrean pekerjaan milik pengguna
squeue -u "$USER"

# Lihat keluaran dan pesan kesalahan (nama berkas mengikuti skrip)
cat flood_job-<JOB_ID>.out
cat flood_job-<JOB_ID>.err
```

Jangan menyalin alamat klaster atau nama pengguna contoh sebagai alamat sungguhan. Gunakan kredensial resmi dari administrator fasilitas.

## 9. Contoh skrip SLURM untuk menjalankan Python

Skrip berikut adalah contoh dasar untuk fasilitas yang memakai SLURM. Skrip menyatakan sumber daya, lokasi keluaran, dan perintah Python. Nama partisi, modul Python, batas waktu, dan kebijakan antrean harus disesuaikan dengan fasilitas yang digunakan.

Skrip ini ditampilkan sebagai teks untuk dipelajari; notebook ini tidak mengirim pekerjaan ke klaster HPC.

In [ ]:
slurm_script = """#!/bin/bash
#SBATCH --job-name=flood-simulation
#SBATCH --output=flood-job-%j.out
#SBATCH --error=flood-job-%j.err
#SBATCH --time=00:10:00
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G

# Sesuaikan baris lingkungan Python dengan panduan fasilitas HPC.
# Contoh pada sistem yang menyediakan modul:
# module load python

python flood_simulation.py
"""

print(slurm_script)

### Contoh cara menyimpan skrip
Jika ingin menyimpan contoh di atas dari notebook, jalankan sel berikut. Berkas yang dihasilkan hanya contoh konfigurasi; ia baru bisa digunakan setelah kode `flood_simulation.py`, lingkungan Python, dan parameter fasilitas disiapkan.

In [ ]:
with open("flood_job.slurm", "w", encoding="utf-8") as f:
    f.write(slurm_script)

print("Contoh skrip disimpan sebagai flood_job.slurm")

## 10. Menghubungkan simulasi dengan kebutuhan lokal dan lintas bidang

HPC adalah infrastruktur, bukan tujuan akhir. Hasil komputasi baru bermanfaat ketika diterjemahkan bersama pengetahuan bidang terkait. Contoh aplikasi yang dibahas dalam webinar meliputi:
- **Kebencanaan:** simulasi banjir dan dukungan perencanaan mitigasi.
- **Pertanian:** pemetaan tanaman dari citra drone atau satelit dan dukungan ketahanan pangan.
- **Kesehatan:** analisis penyebaran penyakit atau simulasi molekuler.
- **Lingkungan:** analisis cuaca, polusi udara, dan PM2.5.
- **Kota dan energi:** simulasi penggunaan energi atau optimasi rute kendaraan listrik.

Untuk menerapkan contoh pada masalah nyata di Indonesia, kolaborasi dengan ahli domain, pemilik data, pemerintah/komunitas, dan pengelola infrastruktur komputasi sangat penting.

## 11. Keterampilan dan praktik yang perlu dipelajari

Untuk mulai menggunakan HPC, mahasiswa dapat membangun keterampilan bertahap:
1. Python dan dasar pemrograman ilmiah.
2. Linux, terminal, direktori, dan pengelolaan berkas.
3. Pengolahan data dan pemahaman format data.
4. Menjalankan skrip, membaca log, serta menelusuri kesalahan.
5. Dasar paralelisasi dan pengukuran performa.
6. Penggunaan sistem antrean seperti SLURM sesuai fasilitas.
7. Dokumentasi, reproduksibilitas, keamanan data, dan kolaborasi lintas disiplin.

Kode yang berjalan di laptop belum tentu langsung berjalan di klaster; lingkungan, dependensi, lokasi data, dan batas sumber daya harus disesuaikan.

## 12. Kesimpulan

Webinar ini menekankan pentingnya menjembatani infrastruktur komputasi global dengan kebutuhan lokal. Implementasi simulasi ini menunjukkan bagaimana suatu masalah dapat dirumuskan menjadi model komputasi, dijalankan pada banyak skenario, divisualisasikan, dan ditafsirkan secara hati-hati.

HPC memungkinkan peneliti menangani data, resolusi, dan jumlah simulasi yang lebih besar. Namun, hasil yang berguna tidak hanya bergantung pada kekuatan komputasi, melainkan juga pada kualitas data, validitas model, pemahaman bidang, kolaborasi, serta kemampuan menjalankan dan mengelola pekerjaan komputasi dengan benar.

### Keterbatasan implementasi
- Data elevasi dan curah hujan adalah sintetis.
- Persamaan indikator genangan bukan model hidrologi yang tervalidasi.
- Notebook berjalan pada komputer biasa dan tidak benar-benar mengakses klaster HPC.
- Contoh SLURM hanya ilustrasi; parameter harus mengikuti aturan fasilitas masing-masing.
- Hasil tidak untuk prediksi banjir nyata, evakuasi, atau keputusan keselamatan.